<a href="https://colab.research.google.com/github/kartikpalan1022-reaper/Fine-Tuning-LLMs-with-Hugging-Face/blob/main/Fine_Tuning_LLMs_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning LLMs with Hugging Face

## Step 1: Installing and importing the libraries

In [1]:
!pip uninstall -y transformers peft trl accelerate bitsandbytes
!pip install transformers==4.46.3 peft==0.13.2 trl==0.12.0 accelerate==1.0.1 bitsandbytes

Found existing installation: transformers 4.46.3
Uninstalling transformers-4.46.3:
  Successfully uninstalled transformers-4.46.3
Found existing installation: peft 0.13.2
Uninstalling peft-0.13.2:
  Successfully uninstalled peft-0.13.2
Found existing installation: trl 0.12.0
Uninstalling trl-0.12.0:
  Successfully uninstalled trl-0.12.0
Found existing installation: accelerate 1.0.1
Uninstalling accelerate-1.0.1:
  Successfully uninstalled accelerate-1.0.1
Found existing installation: bitsandbytes 0.50.2
Uninstalling bitsandbytes-0.50.2:
  Successfully uninstalled bitsandbytes-0.50.2
  Using cached transformers-4.46.3-py3-none-any.whl.metadata (44 kB)
  Using cached peft-0.13.2-py3-none-any.whl.metadata (13 kB)
  Using cached trl-0.12.0-py3-none-any.whl.metadata (10 kB)
  Using cached accelerate-1.0.1-py3-none-any.whl.metadata (19 kB)
  Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached transformers-4.46.3-py3-none-any.whl (10.0 MB)
Using c

In [2]:
!pip install huggingface_hub

In [3]:
import torch
from trl import SFTTrainer
from peft import LoraConfig
from datasets import load_dataset
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, pipeline)

In [4]:
import transformers
import peft
import trl
import accelerate

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 4.46.3
PEFT: 0.13.2
TRL: 0.12.0
Accelerate: 1.0.1


## Step 2: Loading the model

In [7]:
# llama_model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path="aboonaji/llama2finetune-v2",
#                                                   quantization_config = BitsAndBytesConfig(load_in_4bit = True,
#                                                                                            bnb_4bit_compute_dtype=getattr(torch,"float16"),
#                                                                                            bnb_4bit_quant_type="nf4"))
# llama_model.config.use_cache = False
# llama_model.config.pretraining_tp = 1
from transformers import BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

llama_model = AutoModelForCausalLM.from_pretrained(
    "aboonaji/llama2finetune-v2",
    quantization_config=bnb_config,
    device_map="auto"
)

llama_model.config.use_cache = False

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Step 3: Loading the tokenizer

In [8]:
llama_tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path="aboonaji/llama2finetune-v2",
    trust_remote_code=True
)

llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "right"

## Step 4: Setting the training arguments

In [11]:
# training_arguments = TrainingArguments(
#     output_dir="./results",
#     per_device_train_batch_size=4,
#     max_steps=100,
# )
training_arguments = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    max_steps=100,
    fp16=True,
    gradient_checkpointing=True,
)

## Step 5: Creating the Supervised Fine-Tuning trainer

In [12]:
llama_sft_trainer = SFTTrainer(
    model=llama_model,
    args=training_arguments,
    train_dataset=load_dataset(
        "aboonaji/wiki_medical_terms_llam2_format",
        split="train"
    ),
    tokenizer=llama_tokenizer,

    peft_config=LoraConfig(
        r=16,
        lora_alpha=16,
        lora_dropout=0.1,
        task_type="CAUSAL_LM",
    ),

    dataset_text_field="text",
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:309: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
max_steps is given, it will override any value given in num_train_epochs


## Step 6: Training the model

In [13]:
# Warning: To run the training you first need to sign up here: https://wandb.ai/authorize?ref=models
# And then you will directly find in the main page your API key which you will enter in the output below.
llama_sft_trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: palankartik3 (palankartik3-international-institute-of-information-tech) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss


TrainOutput(global_step=100, training_loss=1.6631109619140625, metrics={'train_runtime': 1066.0851, 'train_samples_per_second': 0.375, 'train_steps_per_second': 0.094, 'total_flos': 1.090662998716416e+16, 'train_loss': 1.6631109619140625, 'epoch': 0.058300539279988337})

## Step 7: Chatting with the model